# Reproducing Calvigioni, Fuzik, Le Merre et al. (2023) from dandiset 000473

Paper: Calvigioni D., Fuzik J., Le Merre P. et al., "Esr1+ hypothalamic-habenula neurons shape aversive
states", Nature Neuroscience 26, 1245-1255 (2023), doi:10.1038/s41593-023-01367-8.

Dataset: [DANDI:000473](https://dandiarchive.org/dandiset/000473/0.230417.1502), version 0.230417.1502,
streamed from the archive. The paper's Data availability statement cites this version. A later version,
0.261102.1657, holds the same 25 assets (same blobs and sizes) and differs only in its metadata (it adds
the paper as a related resource), so the cited version is pinned.

Claims: `claims/000473/10.1038-s41593-023-01367-8.yaml`, written from the version of record with its
Methods, figure and Extended Data legends, and the Supplementary Information (Supplementary Tables 1 to
10), and from the paper's code (https://github.com/PierreLeMerre/Esr1_NPX_code at commit 85187f3).

The dandiset holds the Neuropixels recordings of Fig. 4 and Extended Data Figs. 7 to 9: 25 sessions, one
per mouse, with spikes, trials, stimuli and LFP. The ex vivo, anatomy and behavior experiments of Figs. 1
to 3 and 5 are not in it, nor are the pupil videos (one session holds an eye-area trace) or the
optotagging recordings of Extended Data Fig. 1. Outside inputs: the code's session table (genotypes) and
its per-session outputs, read from the repository at the pinned commit.

| Claim | Tested here |
|---|---|
| code-outputs-link | tested (link to the code's outputs) |
| ed7a-unit-counts, fig4g-glm-units, fig4e-opto-modulation-genotypes, ed7bd-latency-similar | tested |
| fig4g-opto-tuning-score, fig4g-tuned-unit-counts, ed8h-waveform-classes, ed8o-conditioned-units, ed8r-comodulated-units | tested |
| ed9a-pca-divergent-block2, fig4h-opto-clusters, ed9bg-state-clusters, fig4i-activity-modes | tested |
| fig4k-decoding-pl, ed9no-decoding-other-regions, fig4jl-pl-esr1 | tested |
| fig4f-pupil-persistent, fig4f-pupil-ctl-unchanged, ed7g-eye-closing-airpuff | blocked, data missing (pupil and eye tracking of 25 mice not in the dandiset) |
| ed1j-optotagged-tonic | blocked, data missing (LHA optotagging recordings not in the dandiset) |
| fig2d-markers | blocked, metadata insufficient (Patch-seq in BioStudies S-BSST1069 has unnamed type codes) |
| fig1l-six-types, fig2fh-topography, fig3f-esr1-rtpa, fig3f-telc-necessary, fig3i-aversive-behaviors, fig3jkl-npy-rearing, fig3mn-persistent-cpa, ed10-female-mild-aversive, fig5f-stress-index-female, fig5g-firing-shift, fig5kl-burst-female-intrinsic | blocked, data missing (not in the dandiset) |

Runtime: about 13 minutes, most of it the decoding and the GLM (with ten seeds of the shifts and two variant periods or saturation series).

In [ ]:
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from scipy import stats
import os, resource, time
os.environ.setdefault("OMP_NUM_THREADS", "1")
import esr1npx as E

t0 = time.time()
assets = E.list_assets()
S = E.load_all(assets)
table = E.code_session_table()
gmap = dict(zip(table.session_id, table.geno_code))
for s in S:
    s["geno_code"] = gmap[s["session_id"]]
    E.select_units(s)
    s["clean"] = E.clean_spikes(s)
    s["clean_lfp"] = E.clean_spikes(s, "LFP")  # variant: the Methods' "LFP saturation" wording
print(f"dandiset {E.DANDISET} version {E.VERSION}: {len(assets)} assets, loaded in "
      f"{time.time() - t0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Data check

What was loaded, per session: the subject metadata, the genotype in the file and in the code's
session table, the trials per block, the units, and the selection of the Methods. Then the checks
of the description against the values: the timing of the stimuli relative to the trial starts,
the light power, the units' quality columns, and whether the spikes share the clock of the trial
table (a response to the tone at the true trial starts against the same measure at shifted starts).

Saturation. The Methods remove spikes within 1 s of "LFP saturation periods". The files hold two
series, 'spike saturation' and 'LFP saturation'. The main analysis uses the 'spike saturation' series,
an assumption: the paper's code reads that series for this step (`RemoveSaturationSpikes` in
`utilities/MS_suppl_functions.py`, +- 1 s), and the 'LFP saturation' series of session 147463 covers
all of its block 3 (printed below), which would leave that block without spikes in every unit. The
'LFP saturation' series is run as a variant through the GLM claims (fig4g-glm-units, fig4e,
fig4g-opto-tuning-score, fig4g-tuned-unit-counts).

In [ ]:
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
rows = []
for s in S:
    tr, u = s["trials"], s["units"]
    bc = tr.Block.value_counts().sort_index()
    rows.append({"session": s["session_id"], "subject": s["subject_id"], "sex": s["sex"], "age": s["age"],
                 "genotype_nwb": s["genotype"], "genotype_code": table.set_index("session_id").code_genotype[s["session_id"]],
                 "opto_site": s["opto_site"][:3], "n_trials": len(tr),
                 "blocks": " ".join(f"{int(b)}:{n}" for b, n in bc.items()),
                 "units": len(u), "in_4_regions": int(u.region.notna().sum()), "selected": int(u.selected.sum()),
                 "rec_s": round(s["t_end"])})
info = pd.DataFrame(rows)
print(info.to_string())
print("\nsessions per code genotype:", info.genotype_code.value_counts().to_dict())
print("sessions per NWB genotype:", info.genotype_nwb.value_counts().to_dict())
print("sex:", info.sex.value_counts().to_dict(), " (the paper states male mice)")
print("behavioral series (processing/behavior):", {s["session_id"][:6]: s.get("behavior") for s in S if s.get("behavior")},
      "; sessions without any:", sum(not s.get("behavior") for s in S))
print("sessions where NWB and code genotype differ:",
      info.loc[info.genotype_nwb.str[:4].str.lower() != info.genotype_code.str[:4].str.lower(), ["session", "genotype_nwb", "genotype_code"]].values.tolist())

In [ ]:
# timing of stimuli relative to trial starts, and stimulus metadata
rows = []
for s in S:
    tr, st = s["trials"], s["stim"]
    on = E.train_onsets(st["OptogeneticSeries"][0])
    b2 = tr[tr.Block == 2].start_time.values
    lag_o = np.array([o - b2[np.searchsorted(b2, o) - 1] for o in on if np.searchsorted(b2, o) > 0])
    lag_o = lag_o[lag_o < 2]
    b4 = tr[tr.Block == 4].start_time.values
    ap = st["air_puff"][0]
    lag_a = np.array([a - b4[np.searchsorted(b4, a) - 1] for a in ap if np.searchsorted(b4, a) > 0])
    lag_a = lag_a[lag_a < 2]
    t123 = tr[tr.Block.isin([1, 2, 3])].start_time.values
    aud = st["AudStim_10kHz"][0]
    matched = np.array([np.min(np.abs(t123 - a)) < 0.005 for a in aud])
    rows.append({"session": s["session_id"][:6], "tone_events": len(aud), "tone_at_trial_start": int(matched.sum()),
                 "block123_trials": len(t123), "light_trains": len(on), "block2_trains": len(lag_o),
                 "light_lag_s": f"{np.median(lag_o):.3f}" if len(lag_o) else "-",
                 "puff_events": len(ap), "puff_lag_s": f"{np.median(lag_a):.3f}" if len(lag_a) else "-",
                 "light_series_max": round(float(st["OptogeneticSeries"][1].max()), 2),
                 "opto_power_col_max": float(tr.opto_power.max()),
                 "puff_dur_col": float(tr.loc[tr.Block == 4, "airpuff_duration"].median()),
                 "trial_dur": float((tr.stop_time - tr.start_time).median())})
print(pd.DataFrame(rows).to_string())
r = []
for s in S:
    tr = s["trials"]; i2 = np.where(tr.Block.values == 2)[0][0]
    on = E.train_onsets(s["stim"]["OptogeneticSeries"][0])
    first = on[on >= tr.start_time.values[i2]][0]
    r.append(s["opto_pulse_ts"][i2, 0] / first)
print("trials table opto_pulse_ts (first block-2 trial, first pulse) over the light series time of that pulse: "
      f"{min(r):.1f} to {max(r):.1f}; the column has no unit or description in the files")

In [ ]:
# units: quality columns, ranges
U = pd.concat([s["units"].assign(session=s["session_id"], geno=s["geno_code"]) for s in S])
print("units:", len(U), " in the four regions:", int(U.region.notna().sum()), " selected:", int(U.selected.sum()))
print("ISI violation count range", U.isi_count.min(), U.isi_count.max(), "; violations / spikes range",
      f"{U.isi_rate.min():.4f} {U.isi_rate.max():.3f}", "; second quality column (false positive rate) range",
      f"{U.isi_fp.min():.4f} {U.isi_fp.max():.3f}")
print("mean rate (spikes / last spike time) range, Hz:", f"{U.mean_fr.min():.3f} {U.mean_fr.max():.2f}")
print("peak-to-trough (ms) range:", f"{U.peak_to_trough.min():.3f} {U.peak_to_trough.max():.3f}",
      "; FS and RS both true:", int((U.FS & U.RS).sum()), "; neither:", int((~U.FS & ~U.RS).sum()))
cols = {tuple(s["unit_columns"]) for s in S}
print("units table columns:", sorted(set().union(*cols)), "; identical in all files:", len(cols) == 1,
      "; a curation label column ('good', KSLabel):", any(c.lower() in ("kslabel", "group", "curation", "label") for c in set().union(*cols)))
print("selected units by region:", U[U.selected].region.value_counts().to_dict())
total = sum(sum(len(s["spikes"][i]) for i in s["clean"]) for s in S)
for series in ("spike", "LFP"):
    st_ = [np.asarray(s["saturation"].get(f"{series} saturation start", []), float) for s in S]
    sp_ = [np.asarray(s["saturation"].get(f"{series} saturation stop", []), float) for s in S]
    removed = sum(sum(len(s["spikes"][i]) - len(c[i]) for i in c) for s, c in ((s, E.clean_spikes(s, series)) for s in S))
    print(f"'{series} saturation' series: {sum(np.isfinite(a).sum() for a in st_)} periods, total {sum(np.nansum(b - a) for a, b in zip(st_, sp_)):.0f} s, "
          f"longest {max(np.nanmax(b - a) if np.isfinite(a).any() else 0 for a, b in zip(st_, sp_)):.0f} s; "
          f"spikes of selected units within 1 s of them: {removed} of {total} ({removed / total:.2%})")
s147 = [s for s in S if s["session_id"].startswith("147463")][0]
b3 = s147["trials"].loc[s147["trials"].Block == 3, "start_time"]
print(f"session 147463: block 3 runs {b3.min():.0f} to {b3.max():.0f} s; its 'LFP saturation' periods include",
      [(round(a, 1), round(b, 1)) for a, b in zip(s147["saturation"]["LFP saturation start"], s147["saturation"]["LFP saturation stop"]) if b - a > 60],
      "and its 'spike saturation' periods", [(round(a, 1), round(b, 1)) for a, b in zip(s147["saturation"]["spike saturation start"], s147["saturation"]["spike saturation stop"]) if b - a > 60])

In [ ]:
# clock check: tone response in blocks 1 and 3 at the true trial starts against starts shifted by -1.5 s (inside the ITI)
rows = []
for s in S:
    on = s["trials"].loc[s["trials"].Block.isin([1, 3]), "start_time"].values
    sel = list(s["clean"].keys())
    if not sel:
        rows.append({"session": s["session_id"][:6], "units": 0}); continue
    def resp(onsets):
        post = np.array([E.aligned_counts(s["clean"][i], onsets, 0, 0.1, 0.1).mean() for i in sel])
        pre = np.array([E.aligned_counts(s["clean"][i], onsets, -0.5, 0, 0.5).mean() / 5 for i in sel])
        return (post - pre).mean() / 0.1, stats.wilcoxon(post, pre).pvalue
    r_true, p_true = resp(on)
    r_shift, p_shift = resp(on - 1.5)
    rows.append({"session": s["session_id"][:6], "units": len(sel), "tone_resp_Hz": round(r_true, 3), "p": p_true,
                 "shifted_resp_Hz": round(r_shift, 3), "p_shifted": p_shift})
clock = pd.DataFrame(rows)
print(clock.to_string())
print("sessions with a significant tone response at true starts (p < 0.01):", int((clock.p < 0.01).sum()),
      "; at shifted starts:", int((clock.p_shifted < 0.01).sum()))

## code-outputs-link: the dandiset's sessions and units against the code's outputs

> The dandiset's 25 sessions are the 25 NWB files of the paper's session table, and the units
> selected from them by the paper's rules are the units listed in the code's per-session GLM
> outputs.

Sessions are matched by subject and date. The code's per-session outputs (`analysis/GLM_fitted/*_tmu.mat`,
`*_tuning_scores.mat`, `*_tuned_bol.mat`, and `analysis/Mean_FR/*_fr.mat`) are read from the repository
at commit 85187f3. Units are selected as in `Fig4g_GLM_tuning.m`: region by the main channel, ISI
violations over spikes < 0.01, mean session rate > 0.1 Hz, and a mean count per 100 ms bin above 0.2.
The code bins counts from files that are not in the repository (`Binned_FR`), so the period they cover
is open; `GLM_restrict = 0` is set, but the same script holds a commented restriction to blocks 1 to 4
(up to 2 s before the first block-5 trial). Both periods are run, with and without the Methods' removal
of spikes near LFP saturation.

In [ ]:
names = {s["session_id"] for s in S}
print("sessions in the dandiset:", len(names), "; in the code's table:", len(table),
      "; matched by subject and date:", len(names & set(table.session_id)))
code = {}
for sid in table.session_id:
    code[sid] = {k: E.code_mat(f"analysis/GLM_fitted/{sid}_{k}.mat") for k in ("tmu", "tuning_scores", "tuned_bol")}
    code[sid]["fr"] = E.code_mat(f"analysis/Mean_FR/{sid}.nwb_fr.mat")
fr_rows = []
for s in S:
    fr = code[s["session_id"]]["fr"]["firing_rates"].ravel()
    u = s["units"]
    ok = (u.n_spikes > 0).values & (fr > 0)
    ratio = u.mean_fr.values[ok] / fr[ok]
    sel_code = (u.region.notna() & (u.isi_rate < 0.01) & (fr > 0.1)).sum()
    fr_rows.append({"session": s["session_id"][:6], "units": len(u), "code_fr_units": len(fr),
                    "ratio_median": round(float(np.median(ratio)), 3), "ratio_5pct": round(float(np.percentile(ratio, 5)), 3),
                    "ratio_95pct": round(float(np.percentile(ratio, 95)), 3),
                    "selected_own_rate": int(u.selected.sum()), "selected_code_rate": int(sel_code)})
fr_tab = pd.DataFrame(fr_rows)
print(fr_tab.to_string())
print("selected with own rate:", fr_tab.selected_own_rate.sum(), "; with the code's Mean_FR rate:", fr_tab.selected_code_rate.sum())

In [ ]:
def fitted_sets(period, clean):
    out = {}
    for s in S:
        u = s["units"]
        t_stop = E.glm_period_end(s) if period == "blocks1to4" else s["t_end"]
        nb = int(round(t_stop / 0.1))
        sp = s["clean"] if clean else s["spikes"]
        out[s["session_id"]] = {int(i) for i in u.index[u.selected]
                                if (sp[int(i)] < nb * 0.1).sum() / nb > 0.2}
    return out
link_rows = []
for period in ("blocks1to4", "session"):
    for clean in (False, True):
        fs = fitted_sets(period, clean)
        same = 0; tot_mine = {}; tot_code = {}; common = 0; only_mine = 0; only_code = 0
        for sid, mine in fs.items():
            t = code[sid]["tmu"]
            tmu = set(t["tmu"].ravel().tolist()) if t is not None else set()
            same += mine == tmu
            g = gmap[sid]
            tot_mine[g] = tot_mine.get(g, 0) + len(mine); tot_code[g] = tot_code.get(g, 0) + len(tmu)
            common += len(mine & tmu); only_mine += len(mine - tmu); only_code += len(tmu - mine)
        link_rows.append({"period": period, "saturation_removed": clean, "sessions_identical": same,
                          "common": common, "only_dandiset": only_mine, "only_code": only_code,
                          **{f"{g}_mine/code": f"{tot_mine[g]}/{tot_code[g]}" for g in E.GENOTYPES},
                          "within_10pct": all(abs(tot_mine[g] - tot_code[g]) <= 0.1 * tot_code[g] for g in E.GENOTYPES)})
link_tab = pd.DataFrame(link_rows)
print(link_tab.to_string())
fs = fitted_sets("blocks1to4", False)
extra = []
for s in S:
    t = code[s["session_id"]]["tmu"]
    tmu = set(t["tmu"].ravel().tolist()) if t is not None else set()
    extra += [(s["session_id"][:6], i, s["units"].at[i, "location"]) for i in sorted(fs[s["session_id"]] - tmu)]
ex = pd.DataFrame(extra, columns=["session", "unit", "location"])
print("units fitted here but absent from the code's outputs, by session and location:")
print(ex.groupby(["session", "location"]).size().to_string())

In [ ]:
# the dandiset's GLM tuning (run below, Methods period reading) against the code's per-unit tuning scores and flags
def code_unit_table():
    rows = []
    for sid, c in code.items():
        if c["tmu"] is None:
            continue
        tmu = c["tmu"]["tmu"].ravel(); ts = c["tuning_scores"]["tun_scores"]; tb = c["tuned_bol"]["tun_bol"]
        for k, i in enumerate(tmu):
            rows.append({"session_id": sid, "unit": int(i), **{f"code_score_{p}": ts[j, k] for j, p in enumerate(E.PREDICTORS)},
                         **{f"code_tuned_{p}": int(tb[j, k]) for j, p in enumerate(E.PREDICTORS)}})
    return pd.DataFrame(rows)
code_units = code_unit_table()
print("units in the code's GLM outputs:", len(code_units), code_units.merge(table, on="session_id").geno_code.value_counts().to_dict())

**Result: partially reproduced, resting on an assumption.** All 25 sessions match the code's session table by subject and date, but the files label two of the code's five C57BL-6J sessions (216300, 216301) as Vglut2-cre. Over blocks 1 to 4 without saturation removal, 18 of 25 sessions give the code's fitted units index for index (4 of them have no units), under the 22 the criterion asks. The per-genotype counts are within 10% of the code's outputs (Ctl 492/482, Vglut2 743/738, Npy 445/406, Esr1 318/311). Over the whole session (`GLM_restrict = 0`) only 4 sessions match and the counts are not within 10%, so the outcome rests on the blocks 1 to 4 period. No unit in the code's outputs is missing here; 61 units fitted here are absent from them, 58 of them on ILA6a or ORBm6a channels. The rate computed here (spike count over the last spike time) is a median 1.03 to 1.09 times the Mean_FR rate, per session; with either rate the selection differs by 5 units. The dandiset's tuning scores agree with the code's (Spearman r 0.95 to 0.99, tuned flags agree for 90 to 92% of units).

## ed7a-unit-counts: units in the analysis

> The analysis holds 4,152 well-isolated units in ACAd, PL, ILA and ORBm from 25 mice (Ctl 1,147,
> Vglut2 1,413, Npy 981, Esr1 611).

Selection as the criterion gives it; genotypes from the code's session table. The rate is the spike count
over the recording (last spike time). Variants: the code's Mean_FR rate, and the files' own genotype labels.

In [ ]:
sel = U[U.selected].copy()
sel["geno_code"] = sel.session.map(gmap)
counts = sel.geno_code.value_counts().reindex(E.GENOTYPES)
expected = {"Ctl": 1147, "Vglut2": 1413, "Npy": 981, "Esr1": 611}
print("selected units, all:", len(sel), "(expected 4152 +- 415.2)")
for g in E.GENOTYPES:
    print(f"  {g}: {counts[g]} (expected {expected[g]} +- {0.1 * expected[g]:.1f}) within: {abs(counts[g] - expected[g]) <= 0.1 * expected[g]}")
print("with the code's Mean_FR rate:", fr_tab.selected_code_rate.sum())
sel["geno_nwb"] = sel.session.map({s["session_id"]: s["geno"] for s in S})
print("by the files' genotype labels:", sel.geno_nwb.value_counts().reindex(E.GENOTYPES).to_dict())
print("mice contributing units:", sel.session.nunique(), "of", len(S), "; per genotype:", sel.groupby("geno_code").session.nunique().to_dict())
print("per session:", sel.groupby("session").size().to_dict())

**Result: reproduced.** 4,159 units pass the selection (4,152 +- 415.2). By genotype: Ctl 1,149, Vglut2 1,415, Npy 984, Esr1 611, all within tolerance, with genotypes from the code's session table. The files' own genotype labels would give Ctl 622 and Vglut2 1,942. Only 21 of the 25 mice contribute units: sessions 128515, 258412, 259112 and 273853 have no unit in the four subregions.

## GLM and tuning scores

The GLM of the Methods, fitted on the dandiset's spikes with spikes within 1 s of the files' 'spike
saturation' periods removed (the code's series; see the data check), over the period from the session
start to 2 s before the first block-5 trial. The Methods do not state the period; this one is an
assumption, taken because the code's fitted units match it and not the whole session (code-outputs-link
above), although `Fig4g_GLM_tuning.m` sets `GLM_restrict = 0`. Variants, both run: the whole session
(`GLM_restrict = 0`), and the 'LFP saturation' series. Units fitted: mean count per 100 ms bin above 0.2
(the code). See the helper for the closed-form fit.

The tuned flags compare each true beta with extreme order statistics of 1,000 shifted betas, so they
depend on the random shifts. The fit is therefore repeated with nine more seeds (1 to 9), and the
claims that count tuned units show the spread over the ten seeds. A null copy of every unit (its counts
rotated by a random 10 to 90% of the period, which breaks their relation to the stimuli) measures the
false-positive rate of the tuned flags under this procedure.

In [ ]:
t1 = time.time()
EXTRA_SEEDS = list(range(1, 10))
glm = E.run_glm(S, period="blocks1to4", clean=True, min_count=0.2, seed=0, extra_seeds=EXTRA_SEEDS, null_seed=99)
glm_sess = E.run_glm(S, period="session", clean=True, min_count=0.2, seed=0)
glm_lfp = E.run_glm(S, period="blocks1to4", clean=True, min_count=0.2, seed=0, spikes_key="clean_lfp")
print(f"GLM runs in {time.time() - t1:.0f} s")
fit = glm[glm.fitted].copy()
fit_lfp = glm_lfp[glm_lfp.fitted].copy()
print("fitted units (Methods period):", len(fit), "; whole-session variant:", int(glm_sess.fitted.sum()),
      "; 'LFP saturation' variant:", len(fit_lfp))
print("bins where two regressors overlap, by session:", glm.groupby("session_id").overlap_bins.first().loc[lambda x: x > 0].to_dict())
print("fitted units with a regressor holding no spike (true data):", int((fit.zero_true > 0).sum()),
      "; shifted fits with an empty regressor:", int(fit.zero_shift.sum()), "of", len(fit) * 4000)
cmp = fit.merge(code_units, on=["session_id", "unit"])
print("units in both the dandiset fit and the code's outputs:", len(cmp))
for p in E.PREDICTORS:
    r = stats.spearmanr(cmp["score_" + p], cmp["code_score_" + p]).statistic
    agree = (cmp["tuned_" + p] == cmp["code_tuned_" + p]).mean()
    print(f"  {p}: Spearman r of tuning scores {r:.3f}; tuned flag agreement {agree:.3f}; "
          f"code +{(cmp['code_tuned_' + p] == 1).sum()} -{(cmp['code_tuned_' + p] == -1).sum()}, "
          f"dandiset +{(cmp['tuned_' + p] == 1).sum()} -{(cmp['tuned_' + p] == -1).sum()}")
nul = fit[["null_tuned_" + p for p in E.PREDICTORS]].values
k_null, n_null = int((nul != 0).sum()), nul.size
ci = stats.binomtest(k_null, n_null).proportion_ci()
print(f"false-positive rate of the tuned flags on the null copies: {k_null} of {n_null} unit-regressor tests "
      f"= {k_null / n_null:.4%} (95% CI {ci.low:.4%} to {ci.high:.4%}); positive {int((nul == 1).sum())}, negative {int((nul == -1).sum())}; "
      "the criterion's chance rate 0.11%")
FP_RATE = k_null / n_null

## fig4g-glm-units: units fitted with the GLM

> 1,945 units were fitted with the GLM (Ctl 482, Vglut2 738, Npy 406, Esr1 319).

Readings, both run: (a) as coded, mean count per 100 ms bin above 0.2; (b) as the code's comment states,
a mean rate above 0.2 Hz (count per bin above 0.02). Counts over the Methods period with saturation spikes
removed; the other periods are shown as variants.

In [ ]:
exp_glm = {"all": 1945, "Ctl": 482, "Vglut2": 738, "Npy": 406, "Esr1": 319}
def glm_counts(df, thr):
    c = df[df.mean_count > thr].geno.value_counts().reindex(E.GENOTYPES).fillna(0).astype(int).to_dict()
    c["all"] = sum(c.values())
    return c
res_glmunits = {}
for name, df, thr in [("(a) count/bin > 0.2, Methods period", glm, 0.2), ("(b) rate > 0.2 Hz, Methods period", glm, 0.02),
                      ("(a) count/bin > 0.2, whole session", glm_sess, 0.2), ("(b) rate > 0.2 Hz, whole session", glm_sess, 0.02),
                      ("(a) count/bin > 0.2, Methods period, 'LFP saturation'", glm_lfp, 0.2)]:
    c = glm_counts(df, thr)
    ok = all(abs(c[k] - v) <= 0.1 * v for k, v in exp_glm.items())
    res_glmunits[name] = (c, ok)
    print(name, {k: c[k] for k in exp_glm}, "all five within 10%:", ok)

**Result: reproduced, resting on an assumption.** Under reading (a) (count per bin above 0.2, blocks 1 to 4), 1,995 units are fitted: Ctl 492, Vglut2 743, Npy 443, Esr1 317. All five counts are within 10% of 1,945, 482, 738, 406 and 319. Reading (b) (0.2 Hz) gives 3,758 and fails. Over the whole session, reading (a) gives 2,195, with Ctl, Npy and Esr1 outside 10%. With the 'LFP saturation' series, reading (a) gives 1,970 (Ctl 492, Vglut2 737, Npy 425, Esr1 316), also within 10%. The outcome rests on the period, the same assumption as for code-outputs-link.

## fig4e-opto-modulation-genotypes: which stimuli modulate mPFC units, per genotype

> Optogenetic activation strongly modulates mPFC units in Vglut2-cre and Esr1-cre mice but not in Npy-cre
> mice, while the auditory stimuli and air puffs modulate mPFC units in all four groups.

Main reading: the fraction of GLM-fitted units tuned (either sign) to the opto regressor, each genotype
against Ctl by Fisher's exact test (two-sided, direction required); the fractions tuned to sound 1 and air
puff against 0.11% by a one-sided binomial test. Second reading: block-2 population PSTH of all selected
units (10 ms bins, z-scored to the 1 s pre-tone baseline), mean over 0.7 to 1.2 s, against 0 by a
Wilcoxon signed-rank test over units.

In [ ]:
def tuned_frac(df, p):
    return df.groupby("geno")["tuned_" + p].apply(lambda x: (x != 0).sum()).reindex(E.GENOTYPES), df.geno.value_counts().reindex(E.GENOTYPES)
res4e = {}
for label, df in [("Methods period", fit), ("whole session", glm_sess[glm_sess.fitted])]:
    k_o, n = tuned_frac(df, "opto")
    print(f"--- {label}")
    rows = []
    for g in E.GENOTYPES:
        r = {"geno": g, "n": n[g], "opto_tuned": k_o[g], "opto_frac": k_o[g] / n[g]}
        if g != "Ctl":
            r["fisher_p_vs_Ctl"] = stats.fisher_exact([[k_o[g], n[g] - k_o[g]], [k_o["Ctl"], n["Ctl"] - k_o["Ctl"]]]).pvalue
        for p in ("sound1", "airpuff"):
            k, _ = tuned_frac(df, p)
            r[p + "_frac"] = k[g] / n[g]
            r[p + "_binom_p"] = stats.binomtest(int(k[g]), int(n[g]), 0.0011, alternative="greater").pvalue
        rows.append(r)
    t = pd.DataFrame(rows); print(t.to_string()); res4e[label] = t
    per = df.groupby(["geno", "session_id"]).tuned_opto.agg(lambda x: f"{(x != 0).sum()}/{len(x)}")
    print("opto-tuned units per mouse:", {g: per.loc[g].tolist() for g in E.GENOTYPES})

In [ ]:
# robustness of the opto comparison: correction for three comparisons, per mouse, seeds, chance rate, saturation series
t4 = res4e["Methods period"].set_index("geno")
print("Fisher p against Ctl, Bonferroni for 3 comparisons:", {g: float(f"{min(1, 3 * t4.at[g, 'fisher_p_vs_Ctl']):.3g}") for g in ("Vglut2", "Npy", "Esr1")})
pm4 = fit.groupby(["geno", "session_id"]).tuned_opto.apply(lambda x: (x != 0).mean())
print("opto-tuned fraction per mouse:", {g: pm4.loc[g].round(3).tolist() for g in E.GENOTYPES})
for g in ("Vglut2", "Npy", "Esr1"):
    print(f"  {g} against Ctl over mice: Mann-Whitney two-sided p = {stats.mannwhitneyu(pm4.loc[g], pm4.loc['Ctl']).pvalue:.3f}, "
          f"one-sided (greater) p = {stats.mannwhitneyu(pm4.loc[g], pm4.loc['Ctl'], alternative='greater').pvalue:.3f} "
          f"({len(pm4.loc[g])} and {len(pm4.loc['Ctl'])} mice)")
def tcol(p, k):
    return f"tuned_{p}" if k == 0 else f"tuned_{p}_s{k}"
n4 = fit.geno.value_counts().reindex(E.GENOTYPES)
rows = []
for k in [0] + EXTRA_SEEDS:
    kk = fit.groupby("geno")[tcol("opto", k)].apply(lambda x: (x != 0).sum()).reindex(E.GENOTYPES)
    r = {"seed": k, **{f"{g}_frac": round(kk[g] / n4[g], 3) for g in E.GENOTYPES}}
    for g in ("Vglut2", "Npy", "Esr1"):
        r[f"{g}_p"] = float(f"{stats.fisher_exact([[kk[g], n4[g] - kk[g]], [kk['Ctl'], n4['Ctl'] - kk['Ctl']]]).pvalue:.3g}")
    rows.append(r)
seed4e = pd.DataFrame(rows); print(seed4e.to_string())
print("seeds with Npy above Ctl at p < 0.05:", int(((seed4e.Npy_p < 0.05) & (seed4e.Npy_frac > seed4e.Ctl_frac)).sum()), "of", len(seed4e),
      "; Npy p range", seed4e.Npy_p.min(), "to", seed4e.Npy_p.max(),
      "; seeds with Vglut2 and Esr1 above Ctl at p < 0.05:", int(((seed4e.Vglut2_p < 0.05) & (seed4e.Esr1_p < 0.05)).sum()))
for p in ("sound1", "airpuff"):
    kk = fit.groupby("geno")["tuned_" + p].apply(lambda x: (x != 0).sum()).reindex(E.GENOTYPES)
    print(f"{p}: largest binomial p against the measured null rate {FP_RATE:.4%} (or 0.25% if larger):",
          f"{max(stats.binomtest(int(kk[g]), int(n4[g]), max(FP_RATE, 0.0025), alternative='greater').pvalue for g in E.GENOTYPES):.3g}")
print("--- 'LFP saturation' variant")
kl = fit_lfp.groupby("geno").tuned_opto.apply(lambda x: (x != 0).sum()).reindex(E.GENOTYPES); nl = fit_lfp.geno.value_counts().reindex(E.GENOTYPES)
for g in E.GENOTYPES:
    pv = stats.fisher_exact([[kl[g], nl[g] - kl[g]], [kl["Ctl"], nl["Ctl"] - kl["Ctl"]]]).pvalue if g != "Ctl" else np.nan
    ks = [int((fit_lfp[fit_lfp.geno == g]["tuned_" + p] != 0).sum()) for p in ("sound1", "airpuff")]
    print(f"  {g}: opto {kl[g]}/{nl[g]} = {kl[g] / nl[g]:.3f}, Fisher p against Ctl {pv:.3g}; sound 1 {ks[0] / nl[g]:.3f}, air puff {ks[1] / nl[g]:.3f}")

In [ ]:
# second reading: block-2 population PSTH, every unit z-scored to its 1 s pre-tone baseline (10 ms bins)
def unit_psth_z(s, i, block, every_other=False, t0=-1.0, t1=3.0, b=0.01):
    on = E.block_onsets(s, block, every_other)
    c = E.aligned_counts(s["clean"][i], on, t0, t1, b).mean(0) / b
    base = c[:int(round(-t0 / b))]
    sd = base.std()
    return (c - base.mean()) / sd if sd > 0 else np.full_like(c, np.nan)
tt = np.arange(-1.0, 3.0, 0.01) + 0.005
win = (tt >= 0.7) & (tt < 1.2)
rows = []
psth_b2 = {}
for g in E.GENOTYPES:
    vals = []
    for s in S:
        if s["geno_code"] != g:
            continue
        for i in s["clean"]:
            z = unit_psth_z(s, i, 2)
            if np.isfinite(z).all():
                vals.append(z)
    v = np.array(vals); psth_b2[g] = v
    m = v[:, win].mean(1)
    rows.append({"geno": g, "units": len(v), "units_dropped_flat_baseline": int(sum(s["units"].selected.sum() for s in S if s["geno_code"] == g) - len(v)),
                 "mean_z_0.7-1.2s": m.mean(), "median": np.median(m), "wilcoxon_p": stats.wilcoxon(m).pvalue})
psth_tab = pd.DataFrame(rows); print(psth_tab.to_string())
fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for a, g in zip(ax, E.GENOTYPES):
    a.plot(tt, psth_b2[g].mean(0)); a.axvspan(0.7, 1.2, color="c", alpha=0.2); a.set_title(f"{g}, block 2"); a.set_xlabel("s from trial start")
ax[0].set_ylabel("mean z")
plt.show()
print("plotted range of mean z:", {g: (round(float(psth_b2[g].mean(0).min()), 2), round(float(psth_b2[g].mean(0).max()), 2)) for g in E.GENOTYPES})

**Result: partially reproduced, resting on an assumption.** The opto-tuned fraction is higher than Ctl (49/492, 10.0%) in Vglut2 (49.7%, Fisher p = 6.9e-52) and Esr1 (34.7%, p = 1.2e-17), and all four genotypes exceed chance for sound 1 (41 to 64%) and air puff (37 to 50%), binomial p below 1e-290 against 0.11%, and below 1e-246 against 0.25%, above the false-positive rate of the tuned flags measured on null copies of the units (0.23%, 95% CI 0.13 to 0.36%). The measured rate is about twice the stated 0.11% because the flags compare each beta with extreme percentiles of 1,000 shifts, as the paper's code does; the observed fractions are far above either. These parts hold under every seed, period and saturation series shown. The part "Npy does not" fails on the criterion's test with seed 0: Npy is also higher than Ctl (64/443, 14.4%, p = 0.044). This is fragile. Over the ten seeds of the shifts, Npy is above Ctl at p < 0.05 in 5 of 10 (p 0.014 to 0.357). With a Bonferroni correction for the three comparisons, p = 0.132. Over mice, the Npy fractions (0.009 to 0.277, 4 mice) do not differ from Ctl (0.037 to 0.211, 5 mice; Mann-Whitney p = 0.905), while Vglut2 (p = 0.008) and Esr1 (p = 0.030) do. Two of the four Npy mice carry the excess (33/201 and 26/94). With the 'LFP saturation' series Npy is 15.5% against 10.4% (p = 0.022). Over the whole session Npy is lower than Ctl (6.7% against 21.4%). With the code's own tuned flags, Npy (14.5%) does not differ from Ctl (11.0%, p = 0.128). In the second reading (block-2 PSTH over 0.7 to 1.2 s), Vglut2 and Esr1 rise (mean z 1.81 and 1.75, medians 0.38 and 0.42). Ctl and Npy have mean z near 0 (-0.010 and 0.0004) with negative medians (-0.14 and -0.15), which are significant against 0 (p = 3.7e-11 and 2.0e-6), so the p-value alone does not separate the genotypes in that reading.

## fig4g-opto-tuning-score: mean absolute tuning score to the light, per genotype

> The mean absolute tuning score to the optogenetic stimulation is higher than in control mice in
> Vglut2-cre and Esr1-cre mice, and not in Npy-cre mice.

Per mouse, the mean absolute opto tuning score over (a) all GLM-fitted units or (b) the units significantly
tuned to the opto regressor; per genotype the mean and s.d. over mice; each genotype against Ctl by a
one-tailed Mann-Whitney test over mice, Bonferroni factor 3.

In [ ]:
exp_ts = {"Ctl": (1.176, 0.258), "Vglut2": (3.337, 0.409), "Npy": (1.126, 0.599), "Esr1": (1.896, 0.612)}
def per_mouse_score(df, p, only_tuned):
    d = df[df["tuned_" + p] != 0] if only_tuned else df
    return d.groupby(["geno", "session_id"])["score_" + p].apply(lambda x: np.abs(x).mean())
res_ts = {}
for label, df in [("Methods period", fit), ("whole session", glm_sess[glm_sess.fitted])]:
    for reading, only in [("(a) all fitted units", False), ("(b) opto-tuned units", True)]:
        print(f"--- {label}, {reading}")
        for p in E.PREDICTORS:
            pm = per_mouse_score(df, p, only)
            rows = []
            for g in E.GENOTYPES:
                v = pm.loc[g].values if g in pm.index.get_level_values(0) else np.array([])
                r = {"geno": g, "mice": len(v), "mean": v.mean() if len(v) else np.nan, "sd": v.std(ddof=1) if len(v) > 1 else np.nan}
                if g != "Ctl" and len(v):
                    r["p_corr"] = min(1.0, 3 * stats.mannwhitneyu(v, pm.loc["Ctl"].values, alternative="greater").pvalue)
                if p == "opto":
                    r["expected"] = f"{exp_ts[g][0]} +- {exp_ts[g][1]}"
                    r["within"] = abs(r["mean"] - exp_ts[g][0]) <= exp_ts[g][1]
                rows.append(r)
            t = pd.DataFrame(rows)
            if p == "opto":
                print(t.to_string()); res_ts[(label, reading)] = t
            else:
                print(p, "corrected p vs Ctl:", {r.geno: round(r.p_corr, 3) for r in t.itertuples() if r.geno != "Ctl"})
print("mice dropped for having no fitted units: 4 (sessions 128515, 258412, 259112, 273853; no units in the four regions)")

**Result: not reproduced, resting on an assumption.** Under reading (a), all four per-mouse means are within tolerance: Ctl 1.31, Vglut2 3.24, Npy 1.44, Esr1 1.93 against 1.176, 3.337, 1.126, 1.896. Vglut2 exceeds Ctl (corrected p = 0.012) and Npy does not (0.83). Esr1 does not exceed Ctl at corrected p < 0.05 (p = 0.11), with 7 Esr1 mice because 3 of the 10 have no units in the four subregions. Over the ten seeds the Esr1 corrected p is 0.072 to 0.11, and with the 'LFP saturation' series 0.072, so no variant reaches 0.05. The code's own tuning scores give the same picture (Esr1 corrected p = 0.159 over the same 7 mice). Reading (b) and the whole-session variant put the means outside tolerance, and Esr1 is not significant under either. No genotype differs from Ctl for the other three regressors (corrected p 0.17 to 1).

## fig4g-tuned-unit-counts: units tuned to the light

> The numbers of units significantly tuned to the optogenetic stimulation are 29 positive and 24 negative in
> Ctl, 293 and 84 in Vglut2, 31 and 31 in Npy, and 92 and 21 in Esr1 mice.

Negative counts drop tuning scores below -30 (the code).

In [ ]:
exp_tc = {"Ctl": (29, 24), "Vglut2": (293, 84), "Npy": (31, 31), "Esr1": (92, 21)}
res_tc = {}
for label, df in [("Methods period", fit), ("whole session", glm_sess[glm_sess.fitted]), ("'LFP saturation'", fit_lfp), ("code's outputs", None)]:
    rows = []
    for g in E.GENOTYPES:
        if df is None:
            d = code_units.merge(table, on="session_id"); d = d[d.geno_code == g]
            pos = int((d.code_tuned_opto == 1).sum()); neg = int(((d.code_tuned_opto == -1) & (d.code_score_opto >= -30)).sum())
        else:
            d = df[df.geno == g]
            pos = int((d.tuned_opto == 1).sum()); neg = int(((d.tuned_opto == -1) & (d.score_opto >= -30)).sum())
        ep, en = exp_tc[g]
        rows.append({"geno": g, "pos": pos, "exp_pos": ep, "pos_within": abs(pos - ep) <= 0.1 * ep,
                     "neg": neg, "exp_neg": en, "neg_within": abs(neg - en) <= 0.1 * en, "both_signs": pos + neg})
    t = pd.DataFrame(rows); res_tc[label] = t
    b = t.set_index("geno").both_signs
    order_ok = min(b["Vglut2"], b["Esr1"]) > max(b["Ctl"], b["Npy"])
    print(f"--- {label}: counts within 10%: {int(t.pos_within.sum() + t.neg_within.sum())} of 8; Vglut2 and Esr1 above Ctl and Npy: {order_ok}")
    print(t.to_string())
# spread over the ten seeds of the shifts (Methods period)
rows = []
for k in [0] + EXTRA_SEEDS:
    tc, sc = ("tuned_opto", "score_opto") if k == 0 else (f"tuned_opto_s{k}", f"score_opto_s{k}")
    r = {"seed": k}; nw = 0; both = {}
    for g in E.GENOTYPES:
        d = fit[fit.geno == g]
        pos = int((d[tc] == 1).sum()); neg = int(((d[tc] == -1) & (d[sc] >= -30)).sum())
        ep, en = exp_tc[g]
        nw += (abs(pos - ep) <= 0.1 * ep) + (abs(neg - en) <= 0.1 * en)
        r[f"{g}_pos"], r[f"{g}_neg"] = pos, neg; both[g] = pos + neg
    r["within_10pct"] = nw; r["order_holds"] = min(both["Vglut2"], both["Esr1"]) > max(both["Ctl"], both["Npy"])
    rows.append(r)
seed_tc = pd.DataFrame(rows); print(seed_tc.to_string())
print("counts within 10% over the ten seeds:", seed_tc.within_10pct.value_counts().sort_index().to_dict(),
      "; seeds with at least 6 of 8:", int((seed_tc.within_10pct >= 6).sum()), "; seeds where the order holds:", int(seed_tc.order_holds.sum()))

In [ ]:
# the same Fisher test on the code's own tuned flags (outside check)
d = code_units.merge(table, on="session_id")
k = d.groupby("geno_code").code_tuned_opto.apply(lambda x: (x != 0).sum()).reindex(E.GENOTYPES); n = d.geno_code.value_counts().reindex(E.GENOTYPES)
for g in ("Vglut2", "Npy", "Esr1"):
    print(f"code's flags, {g}: {k[g]}/{n[g]} = {k[g] / n[g]:.3f} against Ctl {k['Ctl']}/{n['Ctl']} = {k['Ctl'] / n['Ctl']:.3f}, "
          f"Fisher p = {stats.fisher_exact([[k[g], n[g] - k[g]], [k['Ctl'], n['Ctl'] - k['Ctl']]]).pvalue:.3g}")

In [ ]:
# the same test on the code's own tuning scores (outside check)
d = code_units.merge(table, on="session_id")
pm = d.groupby(["geno_code", "session_id"]).code_score_opto.apply(lambda x: np.abs(x).mean())
for g in E.GENOTYPES:
    v = pm.loc[g].values
    pc = min(1, 3 * stats.mannwhitneyu(v, pm.loc["Ctl"].values, alternative="greater").pvalue) if g != "Ctl" else np.nan
    print(f"code's scores, {g}: {len(v)} mice, mean {v.mean():.3f}, sd {v.std(ddof=1):.3f}, corrected p vs Ctl {pc:.3f}")
# seeds (the scores use the shifted betas' mean and s.d.) and the 'LFP saturation' variant, reading (a)
def score_p(df, col):
    pmx = df.groupby(["geno", "session_id"])[col].apply(lambda x: np.abs(x).mean())
    return {g: (round(pmx.loc[g].mean(), 3), min(1, 3 * stats.mannwhitneyu(pmx.loc[g], pmx.loc["Ctl"], alternative="greater").pvalue)) for g in E.GENOTYPES if g != "Ctl"} | {"Ctl": round(pmx.loc["Ctl"].mean(), 3)}
seed_ts = {k: score_p(fit, "score_opto" if k == 0 else f"score_opto_s{k}") for k in [0] + EXTRA_SEEDS}
print("reading (a) over seeds, Esr1 corrected p:", {k: round(v["Esr1"][1], 3) for k, v in seed_ts.items()},
      "; Vglut2 corrected p:", {k: round(v["Vglut2"][1], 3) for k, v in seed_ts.items()})
print("'LFP saturation' variant, reading (a), mean and corrected p:", score_p(fit_lfp, "score_opto"))

**Result: partially reproduced, resting on an assumption.** Under the blocks 1 to 4 period with seed 0, 5 of the 8 counts are within 10%: Vglut2 positive 294 (293), Esr1 positive 90 (92), and the Ctl, Npy and Esr1 negatives 26, 28, 20 (24, 31, 21). Ctl positive 23 (29), Npy positive 36 (31) and Vglut2 negative 75 (84) fall outside. The ordering holds: units tuned either way number 369 in Vglut2 and 110 in Esr1, against 49 in Ctl and 64 in Npy. The count within 10% depends on the seed of the shifts: over ten seeds it is 4 (2 seeds), 5 (4), 6 (2) or 7 (2), so 4 of 10 seeds reach the six the criterion asks for, and the ordering holds in all ten. The outcome stays partially reproduced, the result in seed 0 and in 6 of 10 seeds. The 'LFP saturation' series gives 4 of 8, the whole-session variant none of the eight, and the code's own flags all eight.

## ed8h-waveform-classes: narrow- and wide-spiking units

> Waveform classification gives 564 narrow-spiking putative interneurons and 3,418 wide-spiking putative
> pyramidal units, and both classes include units significantly tuned to the optogenetic stimulation.

Classes from the units table's FS and RS flags among the selected units, with the peak-to-trough rule
(< 0.38 ms narrow, > 0.44 ms wide) as a check.

In [ ]:
print("selected units:", len(sel), "; FS:", int(sel.FS.sum()), "(expected 564 +- 56.4); RS:", int(sel.RS.sum()),
      "(expected 3418 +- 341.8); neither:", int((~sel.FS & ~sel.RS).sum()))
print("peak-to-trough < 0.38 ms:", int((sel.peak_to_trough < 0.38).sum()), "; > 0.44 ms:", int((sel.peak_to_trough > 0.44).sum()),
      "; between:", int(((sel.peak_to_trough >= 0.38) & (sel.peak_to_trough <= 0.44)).sum()))
print("FS units with peak-to-trough < 0.38:", int((sel.FS & (sel.peak_to_trough < 0.38)).sum()),
      "; RS units with peak-to-trough > 0.44:", int((sel.RS & (sel.peak_to_trough > 0.44)).sum()))
fcls = fit.merge(sel[["session", "unit", "FS", "RS"]].rename(columns={"session": "session_id"}), on=["session_id", "unit"])
rows = []
for g in E.GENOTYPES:
    d = fcls[fcls.geno == g]
    rows.append({"geno": g, "WS_pos": int(((d.tuned_opto == 1) & d.RS).sum()), "NS_pos": int(((d.tuned_opto == 1) & d.FS).sum()),
                 "WS_neg": int(((d.tuned_opto == -1) & d.RS).sum()), "NS_neg": int(((d.tuned_opto == -1) & d.FS).sum()),
                 "fitted_WS": int(d.RS.sum()), "fitted_NS": int(d.FS.sum())})
print(pd.DataFrame(rows).to_string())
print("Supplementary Table 7, positive: Vglut2 253 WS and 33 NS; Esr1 77 WS and 13 NS")

**Result: reproduced.** Among the selected units, the FS flag marks 606 units (564 +- 56.4) and the RS flag 3,484 (3,418 +- 341.8). 576 of the 606 FS units have a peak-to-trough below 0.38 ms, and all 3,484 RS units lie above 0.44 ms. Both classes hold opto-tuned GLM units in Vglut2 (213 WS and 71 NS positive) and Esr1 (57 WS and 31 NS positive).

## ed8r-comodulated-units: units comodulated by light and air puff

> Vglut2-cre and Esr1-cre mice have more units comodulated by the optogenetic stimulation and the air puff
> than expected by chance.

Comodulated: primary and secondary tuning are opto and air puff (either order), both significant. Primary
by absolute tuning score (Methods; the code's signed maximum as a variant). Chance from the marginal
probabilities over all fitted units; one-sided binomial test.

In [ ]:
res_comod = {}
for label, signed in [("absolute (Methods)", False), ("signed (code)", True)]:
    p1, p2 = E.primary_secondary(fit, signed=signed)
    tuned = fit[["tuned_" + p for p in E.PREDICTORS]].values != 0
    ar = np.arange(len(fit))
    sig1 = tuned[ar, p1]; sig2 = tuned[ar, p2]
    o, a = E.PREDICTORS.index("opto"), E.PREDICTORS.index("airpuff")
    com = sig1 & sig2 & (((p1 == o) & (p2 == a)) | ((p1 == a) & (p2 == o)))
    P1 = lambda k: np.mean((p1 == k) & sig1); P2 = lambda k: np.mean((p2 == k) & sig2)
    chance = P1(o) * P2(a) + P1(a) * P2(o)
    rows = []
    for g in E.GENOTYPES:
        m = (fit.geno == g).values
        kk, nn = int(com[m].sum()), int(m.sum())
        rows.append({"geno": g, "fitted": nn, "comodulated": kk, "chance_count": round(chance * nn, 2),
                     "binom_p_greater": stats.binomtest(kk, nn, chance, alternative="greater").pvalue})
    t = pd.DataFrame(rows); res_comod[label] = t
    print(f"--- primary tuning {label}; chance rate {chance:.4f}"); print(t.to_string())

**Result: reproduced.** With the Methods' primary tuning (absolute maximum), Vglut2 has 53 comodulated units against 25.65 expected by chance (binomial p = 9.1e-7) and Esr1 has 21 against 10.94 (p = 0.0038). Ctl (3 against 16.99) and Npy (9 against 15.29) fall below chance. The code's signed primary tuning gives the same result (Vglut2 40 against 17.47, Esr1 18 against 7.45).

## ed7bd-latency-similar: latency of the population response to tone and air puff

> The latency of the mPFC population response to the auditory stimulus and to the air puff is similar
> across the four groups of mice.

Population rate (mean over the selected units of a genotype and subregion) in 10 ms bins around the tone
(block-1 trial starts) and the air puff (block-4 air-puff times). Baseline: the window before the event,
as the criterion says, 1 s before the tone and 0.5 s before the air puff (which comes 0.7 s after the
blue-noise onset; the window length is a choice here). As a variant, the 1 s before the trial start (the
Methods' pre-tone baseline) is used for both. Latency: the first bin from the event onset within 500 ms
where the rate exceeds the baseline mean by (a) 3 or (b) 2 baseline s.d. for two consecutive bins.

In [ ]:
def pop_rate(g, region, event, base="pre-event"):
    rates, bases = [], []
    for s in S:
        if s["geno_code"] != g:
            continue
        tr = s["trials"]
        if event == "tone":
            on = tr.loc[tr.Block == 1, "start_time"].values; base_on = on
        else:
            b4 = tr.loc[tr.Block == 4, "start_time"].values
            ap = s["stim"]["air_puff"][0]
            pairs = [(a, b4[np.searchsorted(b4, a) - 1]) for a in ap if np.searchsorted(b4, a) > 0 and a - b4[np.searchsorted(b4, a) - 1] < 2]
            on = np.array([p[0] for p in pairs]); base_on = np.array([p[1] for p in pairs])
        for i in s["clean"]:
            if s["units"].at[i, "region"] != region:
                continue
            rates.append(E.aligned_counts(s["clean"][i], on, 0, 0.5, 0.01).mean(0) / 0.01)
            if base == "pre-event" and event == "airpuff":
                bases.append(E.aligned_counts(s["clean"][i], on, -0.5, 0, 0.01).mean(0) / 0.01)
            else:
                bases.append(E.aligned_counts(s["clean"][i], base_on, -1, 0, 0.01).mean(0) / 0.01)
    if not rates:
        return None, None, 0
    return np.mean(rates, 0), np.mean(bases, 0), len(rates)
def latency(r, b, k):
    thr = b.mean() + k * b.std()
    above = r > thr
    for j in range(len(r) - 1):
        if above[j] and above[j + 1]:
            return j * 10
    return np.nan
res_lat = {}
for base in ("pre-event", "pre-trial"):
    lat_rows = []
    for event in ("tone", "airpuff"):
        for region in E.REGIONS:
            for g in E.GENOTYPES:
                r, b, n = pop_rate(g, region, event, base)
                row = {"event": event, "region": region, "geno": g, "units": n}
                for k in (3, 2):
                    row[f"lat_{k}sd_ms"] = latency(r, b, k) if n else np.nan
                lat_rows.append(row)
    lat = pd.DataFrame(lat_rows)
    print(f"=== baseline {base}")
    print(lat.pivot_table(index=["event", "region"], columns="geno", values=["lat_3sd_ms", "lat_2sd_ms"]).to_string())
    for k in (3, 2):
        ok = {}
        for event in ("tone", "airpuff"):
            sp = lat[lat.event == event].groupby("region")[f"lat_{k}sd_ms"].agg(lambda x: x.max() - x.min() if x.notna().all() else np.nan)
            ok[event] = int((sp <= 20).sum())
            print(f"reading {k} s.d., {event}: spread of latencies across genotypes per region (ms): {sp.to_dict()}")
        res_lat[(base, k)] = ok
        print(f"reading {k} s.d.: regions with spread <= 20 ms, tone {ok['tone']} of 4, air puff {ok['airpuff']} of 4")
print("units per genotype and region:", lat.pivot_table(index="region", columns="geno", values="units").to_dict())

**Result: reproduced, resting on an assumption.** With the pre-event baseline, the tone latencies of the four genotypes lie within 20 ms of one another in 4 of 4 subregions under both thresholds. The air-puff latencies do so in 3 of 4 (PL spreads 30 ms). With the pre-trial baseline (variant), the air-puff latencies agree in only 2 (3 s.d.) or 1 (2 s.d.) of 4 subregions. There the Vglut2 rate in ORBm and PL is already above baseline at puff onset (latency 0 ms). The outcome rests on the baseline window.

## ed8o-conditioned-units: units whose tone and light responses change in block 2

> Opto-modulated mPFC units in Vglut2-cre and Esr1-cre mice include conditioned units, whose response to
> the tone and the light changes over the trials of block 2.

Per selected unit, Wilcoxon signed-rank tests of the mean rate 500 ms before the sound against 500 ms after
the event (block 2: tone and light; block 4: blue noise and air puff), 50 trials, alpha 0.001/4. Readings:
(a) tone and light responses both significant in block 2; (b) light response significant and the tone
response (post minus pre) differing between the first and last 10 block-2 trials by a rank-sum test at
0.001/4. Block-2 trials: every other trial (Methods for heatmaps), the first 50 as a variant.

In [ ]:
ALPHA = 0.001 / 4
def wil(post, pre):
    d = post - pre
    return 1.0 if np.all(d == 0) else stats.wilcoxon(post, pre).pvalue
cond_rows = []
for s in S:
    tr = s["trials"]
    b2_all = tr.loc[tr.Block == 2, "start_time"].values
    b4 = tr.loc[tr.Block == 4, "start_time"].values[:50]
    for i, x in s["clean"].items():
        row = {"session_id": s["session_id"], "unit": i, "geno": s["geno_code"]}
        for sub, b2 in (("eo", b2_all[::2][:50]), ("f50", b2_all[:50])):
            pre = E.aligned_counts(x, b2, -0.5, 0, 0.5)[:, 0] / 0.5
            tone = E.aligned_counts(x, b2, 0, 0.5, 0.5)[:, 0] / 0.5
            light = E.aligned_counts(x, b2, 0.7, 1.2, 0.5)[:, 0] / 0.5
            row[f"p_tone_{sub}"] = wil(tone, pre); row[f"p_light_{sub}"] = wil(light, pre)
            dt = tone - pre
            row[f"p_tone_change_{sub}"] = stats.mannwhitneyu(dt[:10], dt[-10:]).pvalue if np.any(dt[:10] != dt[-10:]) else 1.0
        pre4 = E.aligned_counts(x, b4, -0.5, 0, 0.5)[:, 0] / 0.5
        row["p_noise"] = wil(E.aligned_counts(x, b4, 0, 0.5, 0.5)[:, 0] / 0.5, pre4)
        row["p_puff"] = wil(E.aligned_counts(x, b4, 0.7, 1.2, 0.5)[:, 0] / 0.5, pre4)
        cond_rows.append(row)
cond = pd.DataFrame(cond_rows)
res_cond = {}
for sub in ("eo", "f50"):
    cond[f"a_{sub}"] = (cond[f"p_tone_{sub}"] < ALPHA) & (cond[f"p_light_{sub}"] < ALPHA)
    cond[f"b_{sub}"] = (cond[f"p_light_{sub}"] < ALPHA) & (cond[f"p_tone_change_{sub}"] < ALPHA)
cond["block4"] = (cond.p_noise < ALPHA) & (cond.p_puff < ALPHA)
summ = cond.groupby("geno")[["a_eo", "b_eo", "a_f50", "b_f50", "block4"]].agg(["sum", "mean"]).reindex(E.GENOTYPES)
print(summ.round(4).to_string())
print("units tested:", cond.geno.value_counts().reindex(E.GENOTYPES).to_dict())
print("light-responsive units (every other trial):", cond.groupby("geno").p_light_eo.apply(lambda p: int((p < ALPHA).sum())).reindex(E.GENOTYPES).to_dict())
print("smallest rank-sum p for the tone change (10 vs 10 trials):", f"{cond.p_tone_change_eo.min():.2e}")

**Result: reproduced.** Under reading (a), with every other block-2 trial, 14.6% of Vglut2 units (206) and 13.6% of Esr1 units (83) have significant tone and light responses in block 2, against 1.8% of Ctl units (21) and 4.3% of Npy units (42). The first 50 block-2 trials give the same order (15.5%, 13.3%, 2.7%, 2.8%). Reading (b) finds almost none (at most 2 units in any genotype), so the claim holds under reading (a) only.

## Trial-averaged activity used by the population analyses

Per selected unit, trial-averaged rates in blocks 1 to 4: 10 ms bins from -1 to 3 s around the trial
start (clustering, Methods; PCA uses -0.1 to 3 s and all 100 block-2 trials, Methods), and 50 ms bins from
-2 to 5 s (activity modes, the code's window). For clustering and modes block 2 uses every other trial
(the code).

In [ ]:
t1 = time.time()
P10, P10all, P50, unit_keys = [], [], [], []
for s in S:
    units = list(s["clean"].keys())
    if not units:
        continue
    P10.append(E.block_psths(s, units, -1.0, 3.0, 0.01, every_other_b2=True))
    P10all.append(E.block_psths(s, units, -0.1, 3.0, 0.01, every_other_b2=False))
    P50.append(E.block_psths(s, units, -2.0, 5.0, 0.05, every_other_b2=True))
    unit_keys += [(s["session_id"], i, s["geno_code"], s["units"].at[i, "region"]) for i in units]
P10 = np.concatenate(P10); P10all = np.concatenate(P10all); P50 = np.concatenate(P50)
keys = pd.DataFrame(unit_keys, columns=["session_id", "unit", "geno", "region"])
keys = keys.merge(fit[["session_id", "unit"] + [c for c in fit.columns if c.startswith(("score_", "tuned_"))]], on=["session_id", "unit"], how="left")
keys["fitted"] = keys.score_opto.notna()
print(f"PSTHs in {time.time() - t1:.0f} s; arrays {P10.shape}, {P10all.shape}, {P50.shape}; fitted units among them {int(keys.fitted.sum())}")
print("rate ranges (Hz):", float(P10.min()), float(P10.max()))

## ed9a-pca-divergent-block2: population trajectories per block

> Population trajectories are similar across the four groups in blocks 1, 3 and 4, and diverge in block 2
> for Esr1-cre and Vglut2-cre mice compared with Npy-cre and control mice.

As the Methods and `Extended_Data_Fig9a_PCA.py`: 10 ms bins from -0.1 to 3 s, trial-averaged per block
(all 100 block-2 trials), each unit z-scored over the four blocks (the code), smoothed per block with a
Gaussian of sigma 2 bins, 550 units drawn per genotype, PCA with three components on the 2,200 pooled
units for each block (time bins as observations, as the code), and each genotype's activity projected on
its own loadings. Measure: per block, the mean Euclidean distance between each genotype's trajectory and
the Ctl trajectory over 0.7 to 1.2 s, over 20 draws.

In [ ]:
from scipy.ndimage import gaussian_filter1d
from sklearn.decomposition import PCA
rng = np.random.default_rng(9)
Z = P10all.reshape(len(P10all), -1)
mu, sd = Z.mean(1, keepdims=True), Z.std(1, keepdims=True)
Z = np.nan_to_num((Z - mu) / np.where(sd > 0, sd, np.nan)).reshape(P10all.shape)
Z = gaussian_filter1d(Z, 2, axis=2)
tp = np.arange(-0.1, 3.0, 0.01)[:310] + 0.005
wpc = (tp >= 0.7) & (tp < 1.2)
geno_idx = {g: np.where(keys.geno.values == g)[0] for g in E.GENOTYPES}
dist = np.zeros((20, 4, 4))  # draw, block, genotype
for d in range(20):
    pick = {g: rng.choice(geno_idx[g], 550, replace=False) for g in E.GENOTYPES}
    for b in range(4):
        X = np.vstack([Z[pick[g], b] for g in E.GENOTYPES])  # 2200 x T
        pca = PCA(n_components=3, svd_solver="full").fit(X.T)
        L = pca.components_.T
        traj = {g: Z[pick[g], b].T @ L[k * 550:(k + 1) * 550] for k, g in enumerate(E.GENOTYPES)}
        for k, g in enumerate(E.GENOTYPES):
            dist[d, b, k] = np.linalg.norm(traj[g][wpc] - traj["Ctl"][wpc], axis=1).mean()
mean_d = pd.DataFrame(dist.mean(0), index=[f"block {b + 1}" for b in range(4)], columns=E.GENOTYPES)
print("mean distance to the Ctl trajectory over 0.7-1.2 s (20 draws):"); print(mean_d.round(2).to_string())
cnt = {}
for g in ("Esr1", "Vglut2"):
    k = E.GENOTYPES.index(g)
    ok = (dist[:, 1, k] > dist[:, 1, 2]) & (dist[:, 1, k] > dist[:, 0, k]) & (dist[:, 1, k] > dist[:, 2, k]) & (dist[:, 1, k] > dist[:, 3, k])
    cnt[g] = int(ok.sum())
print("draws in which the block-2 distance exceeds the Npy distance and the genotype's own distance in blocks 1, 3, 4:", cnt)
print("range of distances:", round(float(dist.min()), 2), round(float(dist.max()), 2))

**Result: reproduced.** Over 20 draws, the block-2 distance to the Ctl trajectory is 15.64 for Vglut2 and 11.78 for Esr1, against 1.37 for Npy. In blocks 1, 3 and 4 the Vglut2 and Esr1 distances are 0.56 to 8.61. The condition holds in 20 of 20 draws for both genotypes. The criterion does not test the statement's "similar in block 4". In block 4 all three genotypes are far from Ctl (Vglut2 8.61, Npy 8.41, Esr1 5.77, against 0.56 to 1.51 in blocks 1 and 3), so that part of the statement does not hold on this measure; a question to a reader asks whether the criterion should test it.

## fig4h-opto-clusters: hierarchical clustering of the GLM-fitted units

> Hierarchical clustering of the activity of GLM-fitted units gives clusters of opto-modulated units,
> time-locked or sustained, whose units come from Vglut2-cre and Esr1-cre mice.

As the Methods and the code: trial-averaged rates of the GLM-fitted units (10 ms bins, -1 to 3 s, four
blocks, 1,600 bins), smoothed (Gaussian sigma 0.8 bins, Methods; the code's 5-point gausswin is run as a check: gausswin(5),
alpha 2.5, is a Gaussian of sigma 0.8 bins, so it is the same kernel and not a separate reading), units with 200 or fewer bins above 0.5 Hz dropped (the code's `Active_bins > 200` on the smoothed
rates), z-scored across the four blocks, PCA with the units as variables (`pca(X_all')`), the first 20
loadings as features, Ward linkage, 15 clusters. Opto cluster: the most common primary tuning (absolute
maximum, Methods) of its units is opto.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.ndimage import convolve1d
fk = keys[keys.fitted].reset_index(drop=True)
Pf = P10[keys.fitted.values]
def cluster_units(smooth):
    if smooth == "gauss0.8":
        R = gaussian_filter1d(Pf, 0.8, axis=2, mode="constant")
    else:
        n = np.arange(5) - 2; w = np.exp(-0.5 * (2.5 * n / 2) ** 2); w /= w.sum()  # MATLAB gausswin(5), alpha 2.5
        R = convolve1d(Pf, w, axis=2, mode="constant")
    R = R.reshape(len(R), -1)
    active = (np.abs(R) > 0.5).sum(1)
    keep = active > 200
    X = R[keep]
    X = (X - X.mean(1, keepdims=True)) / X.std(1, keepdims=True)
    pca = PCA(n_components=20, svd_solver="full").fit(X.T)
    feat = pca.components_.T  # units x 20 loadings
    lab = fcluster(linkage(feat, "ward"), 15, criterion="maxclust")
    return keep, lab, X
p1_abs, _ = E.primary_secondary(fk, signed=False)
p1_sig, _ = E.primary_secondary(fk, signed=True)
clus = {}
for smooth in ("gauss0.8", "gausswin5"):
    keep, lab, Xc = cluster_units(smooth)
    d = fk[keep].copy(); d["cluster"] = lab; d["primary"] = np.array(E.PREDICTORS)[p1_abs[keep]]
    d["primary_signed"] = np.array(E.PREDICTORS)[p1_sig[keep]]
    clus[smooth] = (d, Xc)
    within = abs(keep.sum() - 1770) <= 177 or abs(keep.sum() - 1788) <= 178.8
    print(f"=== smoothing {smooth}: units kept {keep.sum()} of {len(fk)} (expected 1770 +- 177 or 1788 +- 178.8); within tolerance: {within}")
    rows = []
    for c, g in d.groupby("cluster"):
        vc = g.primary.value_counts()
        rows.append({"cluster": c, "units": len(g), "top_primary": vc.index[0], "top_frac": round(vc.iloc[0] / len(g), 2),
                     "frac_Vglut2_Esr1": round(g.geno.isin(["Vglut2", "Esr1"]).mean(), 3),
                     **{f"n_{x}": int((g.geno == x).sum()) for x in E.GENOTYPES}})
    ct = pd.DataFrame(rows); clus[smooth] = (d, Xc, ct)
    print(ct.to_string())
    oc = ct[ct.top_primary == "opto"]
    print("opto clusters:", oc.cluster.tolist(), "; of them with >= 90% Vglut2+Esr1 units:", int((oc.frac_Vglut2_Esr1 >= 0.9).sum()))

In [ ]:
# the quality control: active bins (> 0.5 Hz) per unit under the code's rule and other readings
Rs = gaussian_filter1d(Pf, 0.8, axis=2, mode="constant").reshape(len(Pf), -1)  # smoothed trial averages, Hz (code, Methods)
R0 = Pf.reshape(len(Pf), -1)                                                    # unsmoothed trial averages
Rz = (Rs - Rs.mean(1, keepdims=True)) / Rs.std(1, keepdims=True)               # z-scored (the Methods mention z-scoring before the step)
smap = {s["session_id"]: s for s in S}
per_trial = np.zeros(len(fk))
for j, r in enumerate(fk.itertuples()):
    s = smap[r.session_id]
    on = [E.block_onsets(s, b, every_other=(b == 2)) for b in (1, 2, 3, 4)]
    m = min(len(o) for o in on)
    # one trial of each block concatenated (1,600 bins); in a single trial a 10 ms bin is above 0.5 Hz when it holds a spike
    per_trial[j] = sum((E.aligned_counts(s["clean"][r.unit], o[:m], -1.0, 3.0, 0.01) > 0).sum(1) for o in on).mean()
variants = {"smoothed trial average, Hz (code)": (np.abs(Rs) > 0.5).sum(1), "unsmoothed trial average, Hz": (R0 > 0.5).sum(1),
            "smoothed, z-scored, |z| > 0.5": (np.abs(Rz) > 0.5).sum(1), "single trials, mean over trials": per_trial}
for name, a in variants.items():
    q = np.percentile(a, [0, 1, 5, 25, 50])
    print(f"{name}: active bins min {q[0]:.0f}, 1st pct {q[1]:.0f}, 5th {q[2]:.0f}, 25th {q[3]:.0f}, median {q[4]:.0f}; units kept (> 200): {int((a > 200).sum())} of {len(a)}")
a = np.sort(variants["smoothed trial average, Hz (code)"])
print(f"under the code's rule, a cutoff of {a[len(a) - 1770 - 1]:.0f} active bins would keep 1,770 units (the smallest count of active bins is {a[0]:.0f})")
print("mean count per 100 ms bin of the fitted units, minimum:", round(float(fit.mean_count.min()), 3),
      "(0.2 per 100 ms is 2 Hz; one spike over 50 trials in a 10 ms bin is already 2 Hz)")

**Result: partially reproduced.** Two clusters (1 and 2, 64 and 87 units) have opto as their most common primary tuning, and all their units come from Vglut2 and Esr1 mice (100%). The expected count is not met: the quality control keeps all 1,995 fitted units, outside the tolerance of both 1,770 +- 177 and 1,788 +- 178.8. Under the code's rule (more than 200 bins above 0.5 Hz in the smoothed trial averages) the fewest active bins of any fitted unit is 454, so no fitted unit can fail it: the fitted units fire at 2 Hz or more (minimum mean count 0.2 per 100 ms). Unsmoothed averages also keep 1,995, z-scored rates (|z| > 0.5) keep 1,974, and single trials keep 266; none comes near 1,770, which would need a cutoff of 1,308 active bins under the code's rule. The 5-point gausswin of the code is the same kernel as the Methods' Gaussian of sigma 0.8 bins and gives the same clusters, so it is a check and not a separate reading.

## ed9bg-state-clusters: clusters whose baseline changes across blocks

> Five of the clusters ('state' clusters) contain units whose baseline firing rate rises in one block or
> ramps across blocks.

Baseline: mean rate over the 1 s before the sound onset, per unit and block. Per cluster, the six block
pairs by a two-sided Wilcoxon signed-rank test over the cluster's units, Bonferroni for six (the code's
rank-sum test as a variant). A cluster counts when at least one pair is significant and its block means
either change monotonically across blocks 1 to 4, or the block with the highest mean is significantly
above at least one other block (a rise in one block; this reading of the sentence is a choice here).

In [ ]:
from itertools import combinations
res_state = {}
for smooth in ("gauss0.8", "gausswin5"):
    d, Xc, ct = clus[smooth]
    base = Pf[:, :, :100].mean(2)  # raw rates, -1 to 0 s, per block
    rows = []
    for c, g in d.groupby("cluster"):
        bl = base[g.index.values]
        means = bl.mean(0)
        sig_w, sig_r = [], []
        for a, b in combinations(range(4), 2):
            pw = min(1, 6 * stats.wilcoxon(bl[:, a], bl[:, b]).pvalue) if np.any(bl[:, a] != bl[:, b]) else 1
            pr = min(1, 6 * stats.mannwhitneyu(bl[:, a], bl[:, b]).pvalue)
            if pw < 0.05: sig_w.append((a + 1, b + 1))
            if pr < 0.05: sig_r.append((a + 1, b + 1))
        mono = bool(np.all(np.diff(means) > 0) or np.all(np.diff(means) < 0))
        top = int(np.argmax(means)) + 1
        rise = any(top in pr_ and means[top - 1] > means[(pr_[0] if pr_[1] == top else pr_[1]) - 1] for pr_ in sig_w)
        rise_r = any(top in pr_ and means[top - 1] > means[(pr_[0] if pr_[1] == top else pr_[1]) - 1] for pr_ in sig_r)
        rows.append({"cluster": c, "units": len(g), "top_primary": ct.set_index("cluster").top_primary[c],
                     "baseline_means_Hz": np.round(means, 2).tolist(), "signif_pairs_wilcoxon": sig_w,
                     "monotonic": mono, "state_wilcoxon": bool(sig_w) and (mono or rise),
                     "signif_pairs_ranksum": sig_r, "state_ranksum": bool(sig_r) and (mono or rise_r)})
    t = pd.DataFrame(rows); res_state[smooth] = t
    print(f"=== smoothing {smooth}"); print(t.to_string())
    print("clusters meeting the rule: Wilcoxon", int(t.state_wilcoxon.sum()), "; rank-sum (code)", int(t.state_ranksum.sum()),
          "; opto clusters among the Wilcoxon ones:", t[t.state_wilcoxon & (t.top_primary == "opto")].cluster.tolist())

**Result: reproduced.** With the Wilcoxon test, 13 of the 15 clusters have at least one significant block pair with a monotonic or one-block rise in baseline (5 required). The code's rank-sum test gives 9. Both opto clusters are among them. The rule is weak: it rejects only clusters 6 and 8, so it hardly separates state clusters from the rest.

## fig4i-activity-modes: projections on the activity modes

> Projection onto the 'opto OR air puff' mode separates block 2 from block 1 only in Esr1-cre and Vglut2-cre
> mice, and the 'state' mode separates the four blocks completely across time.

As the Methods: trial-averaged rates (50 ms bins, -2 to 5 s, the code's window), each unit's mean rate over
the four blocks subtracted, Gaussian smoothing sigma 0.8 bins. Modes: linear SVM weights (hinge loss, ridge
penalty with C = 1, the equivalent of fitclinear's default lambda = 1/n, uniform classes) trained on
pseudo-population trials of all genotypes (trial k of each session together): 'state', the 1 s before the
sound, block 1 against block 4; 'sound', the 200 ms before against after the tone (block 1); 'opto OR air
puff', 500 ms after light onset (block 2, every other trial) against 500 ms after air-puff onset (block 4).
50 trials each. QR of [state, sound, opto OR air puff] (the code's order) gives W; projections W^T X, per
genotype with W and X restricted to its units. Unit readings: (a) GLM-fitted units tuned to at least one
regressor, (b) all GLM-fitted units, (c) all selected units. The code's windows (2 s baseline, 1 s from 0.7 s)
are a variant.

In [ ]:
from sklearn.svm import LinearSVC
def trial_feats(windows):
    """For each selected unit (keys order), rates in 50 trials per window: windows = list of (block, every_other, t0, t1, event)."""
    out = []
    smap = {s["session_id"]: s for s in S}
    for r in keys.itertuples():
        s = smap[r.session_id]; x = s["clean"][r.unit]
        row = []
        for (blk, eo, t0, t1, ev) in windows:
            on = E.block_onsets(s, blk, eo)[:50]
            if ev == "puff":
                ap = s["stim"]["air_puff"][0]
                on = np.array([ap[np.searchsorted(ap, o)] if np.searchsorted(ap, o) < len(ap) and ap[np.searchsorted(ap, o)] - o < 2 else o + 0.7 for o in on])
                row.append(E.window_rates(x, on, t0, t1))
            else:
                row.append(E.window_rates(x, on, t0, t1))
        out.append(row)
    return np.array(out)  # units x windows x 50
WIN = {"Methods": [(1, False, -1, 0, "t"), (4, False, -1, 0, "t"), (1, False, -0.2, 0, "t"), (1, False, 0, 0.2, "t"),
                   (2, True, 0.7, 1.2, "t"), (4, False, 0, 0.5, "puff")],
       "code": [(1, False, -2, 0, "t"), (4, False, -2, 0, "t"), (1, False, -0.5, 0, "t"), (1, False, 0, 0.5, "t"),
                (2, True, 0.7, 1.7, "t"), (4, False, 0.7, 1.7, "t")]}
F = {k: trial_feats(v) for k, v in WIN.items()}
tm = np.arange(-2, 5, 0.05)[:140] + 0.025
Xm = P50 - P50.mean(axis=(1, 2), keepdims=True)
Xm = gaussian_filter1d(Xm, 0.8, axis=2)
anytuned = keys[[f"tuned_{p}" for p in E.PREDICTORS]].fillna(0).ne(0).any(axis=1).values
readings = {"(a) tuned GLM units": anytuned, "(b) GLM-fitted units": keys.fitted.values, "(c) all selected units": np.ones(len(keys), bool)}
def svm_w(A, B):
    Xs = np.vstack([A.T, B.T]); y = np.r_[np.zeros(A.shape[1]), np.ones(B.shape[1])]
    return LinearSVC(C=1.0, loss="hinge", max_iter=200000, class_weight="balanced", random_state=0).fit(Xs, y).coef_.ravel()
res_modes, res_Q = {}, {}
for wname in ("Methods", "code"):
    Fw = F[wname]
    for rname, m in readings.items():
        f = Fw[m]
        w_state = svm_w(f[:, 0], f[:, 1]); w_sound = svm_w(f[:, 2], f[:, 3]); w_ox = svm_w(f[:, 4], f[:, 5])
        Q, _ = np.linalg.qr(np.c_[w_state, w_sound, w_ox])
        res_Q[(wname, rname)] = (Q, m)
        X = Xm[m]; g_of = keys.geno.values[m]
        proj_all = np.einsum("nk,nbt->kbt", Q, X)
        rng_state = [(proj_all[0, b].min(), proj_all[0, b].max()) for b in range(4)]
        overl = sum(1 for a, b in combinations(range(4), 2) if not (rng_state[a][1] < rng_state[b][0] or rng_state[b][1] < rng_state[a][0]))
        wo = (tm >= 0.7) & (tm < 1.2)
        diffs = {}
        for g in E.GENOTYPES:
            gm = g_of == g
            pg = np.einsum("nk,nbt->kbt", Q[gm], X[gm])
            diffs[g] = float(np.abs(pg[2, 1, wo] - pg[2, 0, wo]).mean())
        cond_i = min(diffs["Esr1"], diffs["Vglut2"]) > max(diffs["Ctl"], diffs["Npy"])
        res_modes[(wname, rname)] = (diffs, overl, cond_i)
        print(f"{wname} windows, {rname}: units {int(m.sum())}; |block 2 - block 1| on 'opto OR air puff' over 0.7-1.2 s:",
              {g: round(v, 3) for g, v in diffs.items()}, f"(i) {cond_i};",
              "state-mode ranges per block:", [tuple(round(float(v), 2) for v in r) for r in rng_state], f"overlapping pairs {overl} of 6, (ii) {overl == 0}")

**Result: reproduced.** Under every unit reading and both window sets, the block 2 minus block 1 difference on the 'opto OR air puff' mode is far larger in Vglut2 and Esr1 than in Ctl and Npy. With the Methods windows and reading (a), the values are Vglut2 199.4, Esr1 26.4, Ctl 1.9, Npy 1.6. The ranges of the four blocks' projections on the 'state' mode do not overlap (0 of 6 pairs) under every reading.

## fig4k-decoding-pl: decoding block identity from baseline activity in PL

> Baseline activity in PL decodes block 1 against block 2 more accurately in Esr1-cre and Vglut2-cre mice
> than in control and Npy-cre mice.

As the Methods and the code: spikes from -2.1 to -0.1 s before trial onset in 100 ms bins, the first 49
trials of blocks 1, 3 and 4 and the first 49 of every other block-2 trial; activity concatenated over the
four blocks and z-scored per unit; units with no baseline spikes dropped; per genotype and subregion with
more than 20 units, logistic regression (lbfgs, max_iter 1000) on 5 to 150 random units in steps of 5,
twofold cross-validation over time bins repeated 25 times (`RepeatedKFold(2, 25, random_state=1)`, the
code), a new unit draw per split. Readings: (a) the accuracy at the largest unit number available in the
subregion for all four genotypes, (b) the mean over the unit numbers available for all four.

As in the code, units are z-scored over all four blocks before the split, and the 100 ms bins of one
trial fall into both training and test folds, so the accuracies include the temporal autocorrelation of
neighbouring bins and slow drifts across blocks. This applies to every genotype alike.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedKFold
dec_units, dec_data = [], []
for s in S:
    tr = s["trials"]
    on = np.concatenate([tr.loc[tr.Block == 1, "start_time"].values[:49], tr.loc[tr.Block == 2, "start_time"].values[::2][:49],
                         tr.loc[tr.Block == 3, "start_time"].values[:49], tr.loc[tr.Block == 4, "start_time"].values[:49]])
    for i, x in s["clean"].items():
        c = E.aligned_counts(x, on, -2.1, -0.1, 0.1).ravel()  # trials x 20 bins, block-major
        if c.std() == 0:
            continue
        dec_data.append((c - c.mean()) / c.std())
        dec_units.append((s["geno_code"], s["units"].at[i, "region"]))
D = np.array(dec_data); du = pd.DataFrame(dec_units, columns=["geno", "region"])
print("units for decoding:", len(D), "of", len(keys), "; time points per block:", D.shape[1] // 4)
print(du.pivot_table(index="region", columns="geno", aggfunc="size").reindex(columns=E.GENOTYPES).to_string())
def decode(pair, seed):
    rng = np.random.default_rng(seed)
    nb = D.shape[1] // 4
    cols = np.r_[np.arange((pair[0] - 1) * nb, pair[0] * nb), np.arange((pair[1] - 1) * nb, pair[1] * nb)]
    y = np.r_[np.zeros(nb), np.ones(nb)]
    rcv = list(RepeatedKFold(n_splits=2, n_repeats=25, random_state=1).split(cols))
    out = {}
    for region in E.REGIONS:
        idx = {g: np.where((du.geno == g) & (du.region == region))[0] for g in E.GENOTYPES}
        nmax = min(150, min(len(v) for v in idx.values()))
        for g in E.GENOTYPES:
            if len(idx[g]) <= 20:
                continue
            for n in range(5, nmax + 1, 5):
                acc = []
                for tr_i, te_i in rcv:
                    pick = rng.choice(idx[g], n, replace=False)
                    X = D[np.ix_(pick, cols)].T
                    clf = LogisticRegression(solver="lbfgs", max_iter=1000).fit(X[tr_i], y[tr_i])
                    acc.append((clf.predict(X[te_i]) == y[te_i]).mean())
                out[(region, g, n)] = (np.mean(acc), np.std(acc))
    return out
t1 = time.time()
DEC = {pair: decode(pair, seed=100 + k) for k, pair in enumerate([(1, 2), (1, 3), (1, 4)])}
print(f"decoding in {time.time() - t1:.0f} s")
def dec_summary(pair):
    rows = []
    for region in E.REGIONS:
        ns = sorted({n for (r, g, n) in DEC[pair] if r == region})
        if not ns:
            continue
        common = [n for n in ns if all((region, g, n) in DEC[pair] for g in E.GENOTYPES)]
        for g in E.GENOTYPES:
            if not common:
                continue
            m_a, s_a = DEC[pair][(region, g, common[-1])]
            m_b = np.mean([DEC[pair][(region, g, n)][0] for n in common]); s_b = np.mean([DEC[pair][(region, g, n)][1] for n in common])
            rows.append({"pair": f"{pair[0]} vs {pair[1]}", "region": region, "geno": g, "n_max_common": common[-1],
                         "acc_a": m_a, "sd_a": s_a, "acc_b": m_b, "sd_b": s_b})
    return pd.DataFrame(rows)
dec_tab = pd.concat([dec_summary(p) for p in DEC])
print(dec_tab.round(3).to_string())

In [ ]:
def beats(t, region, winners, losers, reading):
    d = t[t.region == region].set_index("geno")
    res = {}
    for w in winners:
        res[w] = all(d.at[w, f"acc_{reading}"] - d.at[l, f"acc_{reading}"] > max(d.at[w, f"sd_{reading}"], d.at[l, f"sd_{reading}"]) for l in losers)
    return res
t12 = dec_tab[dec_tab.pair == "1 vs 2"]
for reading in ("a", "b"):
    print(f"reading ({reading}), PL, 1 vs 2:", t12[t12.region == "PL"][["geno", f"acc_{reading}", f"sd_{reading}"]].round(3).values.tolist(),
          "; exceeds Ctl and Npy by more than the larger s.d.:", beats(t12, "PL", ["Esr1", "Vglut2"], ["Ctl", "Npy"], reading))
fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for a, region in zip(ax, E.REGIONS):
    for g in E.GENOTYPES:
        ns = sorted(n for (r, gg, n) in DEC[(1, 2)] if r == region and gg == g)
        if ns:
            a.plot(ns, [DEC[(1, 2)][(region, g, n)][0] for n in ns], label=g)
    a.set_title(f"{region}, block 1 vs 2"); a.axhline(0.5, ls="--", c="k"); a.set_xlabel("units")
ax[0].set_ylabel("accuracy"); ax[-1].legend()
plt.show()
print("accuracy range plotted:", round(min(v[0] for v in DEC[(1, 2)].values()), 3), round(max(v[0] for v in DEC[(1, 2)].values()), 3))

**Result: reproduced.** In PL, block 1 against 2, Esr1 (0.957, s.d. 0.013) and Vglut2 (0.937, s.d. 0.022) exceed Ctl (0.852) and Npy (0.851) by more than the larger s.d. at 150 units (reading (a)). Averaged over 5 to 150 units (reading (b)) they also do (0.876 and 0.854 against 0.764 and 0.769).

## ed9no-decoding-other-regions: no genotype difference elsewhere

> Decoding block identity (1 against 2, 1 against 3, 1 against 4) from the baseline activity of ACAd, ILA and
> ORBm shows no difference between genotypes.

The rule of fig4k-decoding-pl in reverse: a difference is one genotype exceeding another by more than the
larger s.d. over the 50 splits; checked in ACAd, ILA and ORBm for 1 vs 2 and in all four subregions for
1 vs 3 and 1 vs 4.

In [ ]:
res_no = {}
for reading in ("a", "b"):
    diffs = []
    for pair, regions in (("1 vs 2", ["ACAd", "ILA", "ORBm"]), ("1 vs 3", E.REGIONS), ("1 vs 4", E.REGIONS)):
        t = dec_tab[dec_tab.pair == pair]
        for region in regions:
            d = t[t.region == region].set_index("geno")
            for w in d.index:
                for l in d.index:
                    gap = d.at[w, f"acc_{reading}"] - d.at[l, f"acc_{reading}"]
                    if w != l and gap > max(d.at[w, f"sd_{reading}"], d.at[l, f"sd_{reading}"]):
                        diffs.append((pair, region, w, l, round(gap, 3)))
    res_no[reading] = diffs
    print(f"reading ({reading}): {len(diffs)} differences beyond the larger s.d.; with Esr1 the better genotype: {sum(d[2] == 'Esr1' for d in diffs)}:", diffs)

**Result: not reproduced.** Under reading (a), 49 genotype pairs differ by more than the larger s.d.; under reading (b), 41. Among them, block 1 against 2 in ORBm: Esr1 0.879 against Npy 0.714 under (a), and in ACAd Esr1 exceeds every other genotype by 0.063 to 0.091 under (a). Esr1 is the better genotype in most of these differences (counts printed above).

## fig4jl-pl-esr1: where in the mPFC the effects lie

> In Esr1-cre mice the opto-modulation, the state-mode shift and the block decoding are concentrated in the
> prelimbic area, whereas in Vglut2-cre mice they are spread across the mPFC.

Per subregion: (1) the fraction of GLM-fitted units tuned to the opto regressor; (2) the mean over -2 to 5 s
of the block 2 minus block 1 projection on the 'state' mode, W (Methods windows, reading (a) of
fig4i-activity-modes) and X restricted to the genotype's units in the subregion, ranked by its magnitude;
(3) the block 1 vs 2 decoding accuracy of Esr1 minus Ctl (fig4k-decoding-pl, reading (a)).

In [ ]:
Qa, ma = res_Q[("Methods", "(a) tuned GLM units")]
ka = keys[ma].reset_index(drop=True); Xa = Xm[ma]
m1 = fit.groupby(["geno", "region"]).tuned_opto.apply(lambda x: (x != 0).mean()).unstack().reindex(index=E.GENOTYPES, columns=E.REGIONS)
m2 = pd.DataFrame(index=E.GENOTYPES, columns=E.REGIONS, dtype=float)
for g in E.GENOTYPES:
    for r in E.REGIONS:
        sel_ = ((ka.geno == g) & (ka.region == r)).values
        if sel_.sum() == 0:
            continue
        pg = np.einsum("nk,nbt->kbt", Qa[sel_], Xa[sel_])
        m2.loc[g, r] = float((pg[0, 1] - pg[0, 0]).mean())
t12a = dec_tab[dec_tab.pair == "1 vs 2"].pivot(index="region", columns="geno", values="acc_a")
m3 = (t12a["Esr1"] - t12a["Ctl"]).reindex(E.REGIONS)
print("(1) fraction opto-tuned:"); print(m1.round(3).to_string())
print("(2) block 2 - block 1 on the state mode (mean over -2 to 5 s):"); print(m2.round(2).to_string())
print("(3) decoding 1 vs 2, Esr1 minus Ctl:", m3.round(3).to_dict())
first = {"(1)": m1.loc["Esr1"].idxmax(), "(2)": m2.loc["Esr1"].abs().idxmax(), "(3)": m3.idxmax()}
n_pl = sum(v == "PL" for v in first.values())
v_spread = int((m1.loc["Vglut2"] > m1.loc["Ctl"].max()).sum())
print("Esr1, subregion ranked first per measure:", first, f"-> PL first in {n_pl} of 3")
print(f"Vglut2: subregions with opto-tuned fraction above the Ctl maximum ({m1.loc['Ctl'].max():.3f}): {v_spread} of 4")
print("fitted units per genotype and subregion:", fit.groupby(["geno", "region"]).size().unstack().reindex(index=E.GENOTYPES, columns=E.REGIONS).to_dict("index"))
m2c = (m2.loc["Esr1"] - m2.loc["Ctl"]).astype(float)
print("(2) referenced to Ctl, Esr1 minus Ctl:", m2c.round(2).to_dict(), "-> largest magnitude:", m2c.abs().idxmax())
rng_jl = np.random.default_rng(12)
fe = fit[fit.geno == "Esr1"]
tun = {r: (fe[fe.region == r].tuned_opto.values != 0) for r in E.REGIONS}
first_pl = 0
for _ in range(1000):
    fr = {r: rng_jl.choice(v, len(v)).mean() for r, v in tun.items() if len(v)}
    first_pl += max(fr, key=fr.get) == "PL"
print(f"(1) Esr1, bootstrap over units within each subregion (1,000 draws): PL first in {first_pl / 10:.1f}% of draws;",
      "units per subregion:", {r: len(v) for r, v in tun.items()})
print("(1) Esr1, subregion first over the ten seeds:", {k: fe.groupby("region")[tcol("opto", k)].apply(lambda x: (x != 0).mean()).idxmax() for k in [0] + EXTRA_SEEDS})

**Result: reproduced.** In Esr1, PL ranks first on all three measures with seed 0: opto-tuned fraction 0.413, against ILA 0.412 on 17 units; state-mode shift -9.01; decoding gain over Ctl 0.104. In Vglut2, the opto-tuned fraction is above the Ctl maximum (0.115) in 4 of 4 subregions (0.39 to 0.61). The evidence for PL is weak. Measure (1) is a tie: PL is first in 51.7% of bootstrap draws over units and in 7 of 10 seeds. Measure (3) separates PL (0.104) from ORBm (0.099) by less than the decoding s.d. (about 0.02). Measure (2) referenced to Ctl still ranks PL first (Esr1 minus Ctl -2.54, against -2.01 in ORBm), but Ctl's own PL shift (-6.47) is close to Esr1's (-9.01). The rule needs two of three measures, so it holds with measures (2) and (3) whichever subregion wins measure (1). ed9no-decoding-other-regions finds Esr1 decoding as well in ACAd, ILA and ORBm, which does not fit a PL-specific effect.

## What the files hold for the blocked claims

The verifier asked for the outputs behind three blocked outcomes: the locations of the units outside
the four subregions (ed1j-optotagged-tonic, which needs LHA recordings), the files of the Patch-seq
deposit (fig2d-markers), and what the one session with eye data (128514, an Npy mouse) holds
(fig4f-pupil-persistent, ed7g-eye-closing-airpuff). None of these is a test of the claim.

In [ ]:
# ed1j: locations of all units outside ACAd, PL, ILA and ORBm, by genotype
out = U[U.region.isna()]
print("units outside the four subregions:", len(out), "of", len(U))
loc_g = out.groupby(["geno", "location"]).size()
for g in E.GENOTYPES:
    print(f"  {g}: {int(loc_g.loc[g].sum())} units; most common labels:", loc_g.loc[g].sort_values(ascending=False).head(8).to_dict())
lh = out[out.location.str.contains("LH|hypoth|Hypoth", regex=True)]
print("units labelled with LHA, LHb or another LH label:", len(lh), lh.groupby(["geno", "location"]).size().to_dict())

In [ ]:
# fig2d: the files of BioStudies S-BSST1069 and the type codes in its metadata
import requests, io, gzip
info = requests.get("https://www.ebi.ac.uk/biostudies/api/v1/studies/S-BSST1069", timeout=60).json()
def files_of(x):
    if isinstance(x, dict):
        if "path" in x and "size" in x:
            yield x
        for v in x.values():
            yield from files_of(v)
    elif isinstance(x, list):
        for v in x:
            yield from files_of(v)
for fl in files_of(info):
    print(f"  {fl['path']}: {fl['size'] / 1e6:.1f} MB,", {a["name"]: a["value"] for a in fl.get("attributes", [])})
r = requests.get("https://ftp.ebi.ac.uk/biostudies/fire/S-BSST/069/S-BSST1069/Files/metadata.tsv.gz", timeout=60); r.raise_for_status()
meta = pd.read_csv(io.BytesIO(gzip.decompress(r.content)), sep="\t")
print("metadata.tsv.gz columns:", list(meta.columns), "; cells:", len(meta))
print("Ephys_Category counts:", meta.Ephys_Category.value_counts(dropna=False).sort_index().to_dict())
print("the paper's per-type counts for the Patch-seq neurons (Extended Data Fig. 3l legend): FA-Bk 12, Burst + RS-N 77, LS-N 35, LS-W 28")

In [ ]:
# fig4f, ed7g: the behavior series of session 128514 (Npy), described per block
s14 = [s for s in S if s["session_id"].startswith("128514")][0]
B = E.load_behavior([a for a in assets if "128514" in a["path"]][0])
print("series:", {k: (len(v[0]), f"{v[0][0]:.1f} to {v[0][-1]:.1f} s", f"median step {np.median(np.diff(v[0])) * 1e3:.1f} ms") for k, v in B.items()})
te, ea = B["eye_area"]; tb, bl = B["blink"]
def win_mean(t, x, on, a, b):
    i0, i1 = np.searchsorted(t, on + a), np.searchsorted(t, on + b)
    return np.array([x[i:j].mean() if j > i else np.nan for i, j in zip(i0, i1)])
tr14 = s14["trials"]
rows = []
for blk in (1, 2, 3, 4):
    on = tr14.loc[tr14.Block == blk, "start_time"].values
    ev = on.copy()
    if blk == 4:
        ap = s14["stim"]["air_puff"][0]; ev = np.array([ap[np.searchsorted(ap, o)] if np.searchsorted(ap, o) < len(ap) else np.nan for o in on])
    elif blk == 2:
        ev = on + 0.7
    pre = win_mean(te, ea, on, -1, 0); post = win_mean(te, ea, ev, 0, 0.5)
    ratio = post / pre
    rows.append({"block": blk, "trials": len(on), "eye_area_pre_trial_median": round(float(np.nanmedian(pre)), 1),
                 "event": {1: "tone", 2: "light", 3: "tone", 4: "air puff"}[blk],
                 "post_over_pre_median": round(float(np.nanmedian(ratio)), 3),
                 "trials_post_below_half_pre": int(np.nansum(ratio < 0.5)),
                 "blink_post_minus_pre_median": round(float(np.nanmedian(win_mean(tb, bl, ev, 0, 0.5) - win_mean(tb, bl, on, -1, 0))), 1)})
print(pd.DataFrame(rows).to_string())
print("eye_area description in the file: 'smoothed pupil trace extracted with Facemap algorithm form the pupil movies' (unit pixel^2);",
      "one mouse, so no genotype comparison and no test")

**Result: these outputs confirm the blocks.** None of the 5,046 units outside the four subregions carries an LHA, LHb or other LH label; the most common labels are olfactory, orbital, motor and striatal areas (AON, DP, ORBvl, MOs, ACB), so the files hold no LHA optotagging recording (ed1j-optotagged-tonic). BioStudies S-BSST1069 holds four files: the 40.7 GB raw reads, counts, TPM, and a metadata table whose only type column, Ephys_Category, holds codes 1 to 6 (12, 45, 38, 31, 28 and 9 cells, 6 without a code) with no names. If the codes followed the paper's order of the types (FA-Bk, Burst, RS-N, LS-N, LS-W, RS-W), FA-Bk (12) and LS-W (28) would match the paper's counts, but Burst + RS-N (83) and LS-N (31) would not match 77 and 35, so the counts do not settle a mapping and fig2d-markers stays blocked. Session 128514 (Npy) holds blink, face-motion and eye_area series at 50 Hz; the file describes eye_area as a smoothed pupil trace. In that one mouse the eye_area falls below half its pre-trial value within 0.5 s in 7 of 51 air-puff trials, against 2 of 50 tone trials in block 1. One mouse of one genotype allows no comparison between genotypes, so fig4f-pupil-persistent, fig4f-pupil-ctl-unchanged and ed7g-eye-closing-airpuff stay blocked.

## Summary

| Claim | Outcome |
|---|---|
| code-outputs-link | partially reproduced (assumption: GLM period) |
| ed7a-unit-counts | reproduced |
| fig4g-glm-units | reproduced (assumption: GLM period) |
| fig4e-opto-modulation-genotypes | partially reproduced (assumption: GLM period) |
| fig4g-opto-tuning-score | not reproduced (assumption: GLM period) |
| fig4g-tuned-unit-counts | partially reproduced (assumption: GLM period) |
| ed8h-waveform-classes | reproduced |
| ed8r-comodulated-units | reproduced |
| ed7bd-latency-similar | reproduced (assumption: baseline window) |
| ed8o-conditioned-units | reproduced |
| ed9a-pca-divergent-block2 | reproduced |
| fig4h-opto-clusters | partially reproduced |
| ed9bg-state-clusters | reproduced |
| fig4i-activity-modes | reproduced |
| fig4k-decoding-pl | reproduced |
| ed9no-decoding-other-regions | not reproduced |
| fig4jl-pl-esr1 | reproduced |
| 3 pupil and eye claims, the optotagging claim, 11 ex vivo, anatomy and behavior claims | blocked, data missing |
| fig2d-markers | blocked, metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's: the GLM is fitted over the period from the session start to 2 s
before block 5 (the code's commented restriction, which its outputs match), not over the whole session as
`GLM_restrict = 0` states, an assumption with the whole session shown as a variant; the tuned flags are
shown over ten seeds of the shifts, as several counts depend on the seed; its fit uses the closed form of a Poisson GLM with exclusive regressors, exact
for this design; spikes within 1 s of the files' 'spike saturation' periods are removed (the code's series;
the Methods say LFP saturation, run as a variant through the GLM claims); genotypes come from the code's session table; several detection rules
(latency, conditioned units, state clusters, comodulation chance test) are the criteria's choices where the
paper gives none.

What was found about the dataset: two control sessions (216300, 216301) are labelled Vglut2-cre in the
files; 10 of 25 mice are female in the files, where the paper states male mice; four sessions have no unit
in the four subregions, so 21 mice contribute; the light power column is 0 throughout and the light series
peaks at 0.9 to 3.3 (in "mW"), where the paper states 6 mW; the files hold blocks 9 to 12 (light at 5 to
20 Hz) that the paper does not describe, and some sessions have extra blocks or trials (block 0, 100 trials
in block 1 of 259112); `opto_pulse_ts` is in an undocumented unit; three sessions hold tone events outside
the trial table; one session's 'LFP saturation' series spans 19.5 min including all of block 3; the
air-puff duration column says 20 ms, where the paper says 50 ms; the units table has no curation
label, so the Methods' 'good' units cannot be checked; the optogenetic site is LHA in all files,
where the paper describes fibers over the LHb for terminal stimulation.